# Results DHR26

**Table of contents**<a id='toc0_'></a>    
- 1. [Imports](#toc1_)    
- 2. [Settings](#toc2_)    
- 3. [Load](#toc3_)    
  - 3.1. [Hyperparameters](#toc3_1_)    
- 4. [Convergence plot](#toc4_)    
  - 4.1. [No backward](#toc4_1_)    
  - 4.2. [Comparative statics: multiple NNs + FOC](#toc4_2_)    
  - 4.3. [Comparative statics: exploration noise](#toc4_3_)    
- 5. [Euler errors](#toc5_)    
- 6. [Life-cycle profiles](#toc6_)    
  - 6.1. [1D](#toc6_1_)    
  - 6.2. [2D](#toc6_2_)    
  - 6.3. [Table: Moments](#toc6_3_)    
  - 6.4. [Tables: Correlations](#toc6_4_)    
- 7. [Cross-section](#toc7_)    
  - 7.1. [Scatter](#toc7_1_)    
  - 7.2. [Histogram](#toc7_2_)    
    - 7.2.1. [Discrete choices](#toc7_2_1_)    
    - 7.2.2. [1D](#toc7_2_2_)    
    - 7.2.3. [2D](#toc7_2_3_)    
    - 7.2.4. [Agreement figure](#toc7_2_4_)    

<!-- vscode-jupyter-toc-config
	numbering=true
	anchor=true
	flat=false
	minLevel=2
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

## 1. <a id='toc1_'></a>[Imports](#toc0_)

In [28]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [29]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE' # without this python may crash when plotting from matplotlib
import numpy as np
import torch
import pandas as pd
import re
import torch 

import matplotlib.pyplot as plt
colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
plt.rcParams.update({'axes.grid':True,'grid.color': 'black','grid.alpha':'0.25','grid.linestyle':'--'})
plt.rcParams.update({'font.size':14})
plt.rcParams.update({'font.family':'serif'})

In [30]:
import EconDLSolvers

In [31]:
import sys
parent_directory = os.path.abspath('..')
sys.path.append(parent_directory)

from plot_funcs import load_all, train_specs, get_train_specs_rows, compute_transfer, convergence_plot

## 2. <a id='toc2_'></a>[Settings](#toc0_)

In [32]:
Ds = [1,2]
do_display = True

algonames = ['sigmoid','softmax']

folder_load = '../output/'
folder_save = '../output/results/nonconvex/'

## 3. <a id='toc3_'></a>[Load](#toc0_)

In [33]:
models = load_all(folder_load,'NonConvexDurablesModel')

DL 1D backward
DL 1D sigmoid
DL 1D sigmoidFOC
DL 1D sigmoidNNs1
DL 1D sigmoidaltbeta
DL 1D sigmoidepssigma0
DL 1D sigmoidepssigma00expnoise10
DL 1D sigmoidepssigma01expnoise105
DL 1D sigmoidepssigma02expnoise11
DL 1D softmax
DL 2D backward
DL 2D sigmoid
DL 2D softmax
DP 1D
DP 1D altbeta
DP 2D


To numpy:

In [34]:
for model in models:
    sim = models[model].sim
    for name in list(vars(sim)): 
        v = getattr(sim, name)
        if torch.is_tensor(v):
            setattr(sim, name, v.detach().cpu().numpy())

Basic info:

In [35]:
for key,model in models.items():
    print(key)
    omega = model.par.omega if key[0] == 'DP' else model.par.omega.numpy()
    print(f' par.omega = {omega}')
    print(f' par.d_ubar = {model.par.d_ubar}')
    if model.par.D == 1:
        print(f' sim.m.max() = {model.sim.states[0].max():.2f}')
        print(f' sim.p.max() = {model.sim.states[1].max():.2f}')
        print(f' sim.n.max() = {model.sim.states[2].max():.2f}')
    else:
        print(f' sim.m.max() = {model.sim.states[0].max():.2f}')
        print(f' sim.p.max() = {model.sim.states[1].max():.2f}')
        print(f' sim.n1.max() = {model.sim.states[2].max():.2f}')
        print(f' sim.n2.max() = {model.sim.states[3].max():.2f}')

    print('')

('DL', '1D', 'backward')
 par.omega = [0.2]
 par.d_ubar = tensor([0.0100])
 sim.m.max() = 1.63
 sim.p.max() = 2.06
 sim.n.max() = 3.14

('DL', '1D', 'sigmoid')
 par.omega = [0.2]
 par.d_ubar = tensor([0.0100])
 sim.m.max() = 1.63
 sim.p.max() = 2.06
 sim.n.max() = 3.14

('DL', '1D', 'sigmoidFOC')
 par.omega = [0.2]
 par.d_ubar = tensor([0.0100])
 sim.m.max() = 1.63
 sim.p.max() = 2.06
 sim.n.max() = 3.15

('DL', '1D', 'sigmoidNNs1')
 par.omega = [0.2]
 par.d_ubar = tensor([0.0100])
 sim.m.max() = 1.63
 sim.p.max() = 2.06
 sim.n.max() = 3.11

('DL', '1D', 'sigmoidaltbeta')
 par.omega = [0.2]
 par.d_ubar = tensor([0.0100])
 sim.m.max() = 1.63
 sim.p.max() = 2.06
 sim.n.max() = 3.18

('DL', '1D', 'sigmoidepssigma0')
 par.omega = [0.2]
 par.d_ubar = tensor([0.0100])
 sim.m.max() = 1.63
 sim.p.max() = 2.06
 sim.n.max() = 3.16

('DL', '1D', 'sigmoidepssigma00expnoise10')
 par.omega = [0.2]
 par.d_ubar = tensor([0.0100])
 sim.m.max() = 1.63
 sim.p.max() = 2.06
 sim.n.max() = 3.09

('DL', '1D'

### 3.1. <a id='toc3_1_'></a>[Hyperparameters](#toc0_)

In [36]:
do_display = False
for D in Ds:
    if do_display: print(f'D = {D}')
    models_ = {k:v for k,v in models.items() if int(k[1][0]) == D and len(k) == 3 and k[2] == 'sigmoid'}
    train_specs(models_,do_display=do_display,folder=folder_save,filename=f'NonConvexDurablesModel_train_specs_{D}D')

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

In [37]:
parameters = set([k for k,l in get_train_specs_rows(VPD=True)])
latex_strings = dict(get_train_specs_rows(VPD=True))

# remove unimportant hyperparameters (not in table)
parameters -= {"Delta_time", 
         "Delta_epoch_policy", 
         "Delta_epoch_value", 
         "Delta_transfer", 
         "FOC_weight_pol", 
         "FOC_weight_val", 
         "K", 
         "K_time_min",
         "NN_use_best",
         "NN_use_best",
         "N_sample_policy_loss",
         "N_target_batches",
         "Nneurons_policy_t",
         "Nepochs_policy",
         "Nepochs_policy_t",
         "Nepochs_value",
         "Nepochs_value_t",
         "Ngpus",
         "Ninputs_aux",
         "Ninputs_time",
         "Ninputs_time_aux",
         "Nneurons_value_t",
         "backward",
         "budget_shares",
         "clip_grad_policy",
         "clip_grad_value",
         "convergence_plot",
         "do_sim_eps",
         "dtype",
         "epoch_policy_min",
         "epoch_termination",
         "epoch_use_best",
         "epoch_value_min",
         "eq_w",
         "i_t_index",
         "input_transformation",
         "learning_rate_policy_decay_t",
         "learning_rate_policy_schedule",
         "learning_rate_value_decay_t",
         "learning_rate_value_schedule",
         "manual_init_policy",
         "max_actions",
         "min_actions",
         "only_initial_states_and_shocks",
         "redraw_mc",
         "store_actions",
         "store_pd",
         "store_reward",
         "target_value_in_policy",
         "tau_schedule",
         "tau_final",
         "tau",
         "terminal_actions_known",
         "terminate_on_policy_loss",
         "learning_rate_policy_t",
         "learning_rate_value_t",
         "track_sim_policy_loss",
         "update_numint_weights",
         "use_FOC",
         "use_quad",
         "use_simult_in_backward",
         "use_target_policy",
         "use_target_value",
         "value_weight_pol",
         "value_weight_val",
         "time_input_type"
         }


models_ = {}
models_[('DL', '1D', 'sigmoid')] = models[('DL', '1D', 'sigmoid')]
models_[('DL', '2D', 'sigmoid')] = models[('DL', '2D', 'sigmoid')]


In [38]:
rows = EconDLSolvers.figs.get_all_hyperparams(models_)
rows = rows.intersection(rows)

cols = ['DL-1D', 'DL-2D']
df = pd.DataFrame(columns=cols)

for k in sorted(list(parameters)):
    print(k)
    name = latex_strings[k] if k in latex_strings else k
    val_1D = getattr(models[('DL', '1D', 'sigmoid')].train, k)
    val_2D = getattr(models[('DL', '2D', 'sigmoid')].train, k)
    if k == 'N':
        val_1D = int(val_1D)
        val_2D = int(val_2D)
    
    if isinstance(val_1D, torch.Tensor):
        val_1D = val_1D.detach().cpu().numpy()[0]
        val_2D = val_2D.detach().cpu().numpy()[0]
    if isinstance(val_1D, list):
        val_1D = val_1D[0]
        val_2D = val_2D[0]

    df.loc[name] = (val_1D, val_2D)

K_time
N
NFOC_targets
NN_init_std
N_value_NN
Nneurons_policy
Nneurons_value
Nnumint
batch_size
buffer_memory
epsilon_sigma
epsilon_sigma_decay
epsilon_sigma_min
learning_rate_policy
learning_rate_policy_decay
learning_rate_policy_min
learning_rate_value
learning_rate_value_decay
learning_rate_value_min
policy_activation_final
policy_activation_intermediate
sim_R_freq
start_train_policy
tol_policy_loss
value_activation_intermediate


In [39]:
df.to_latex(folder_save + "NonConvexDurablesModel_hyperparameters.tex",
    escape=False,      # if your index/values already contain LaTeX (e.g. $\beta$)
    float_format="%.3f",
    label="tab:hyperparameters_nonconvex_durables",
)

## 4. <a id='toc4_'></a>[Convergence plot](#toc0_)

In [40]:
import torch

for key in vars(models[('DP', '1D')].par).keys():
    par_DP = getattr(models[('DP', '1D')].par, key)
    if hasattr(models[('DL', '1D', 'softmax')].par, key):
        par_DL = getattr(models[('DL', '1D', 'softmax')].par, key)
        
        if isinstance(par_DP, torch.Tensor) or isinstance(par_DL, torch.Tensor):
            assert torch.equal(torch.as_tensor(par_DP), torch.as_tensor(par_DL)), f"Mismatch on {key}"
        else:
            assert par_DP == par_DL, f"Mismatch on {key}"

In [41]:
models[('DP', '1D')].vfi

namespace(Nm=150,
          Na=200,
          Np=100,
          Nn=150,
          Nx=150,
          m_min=1e-08,
          m_max=5.0,
          a_max=6.0,
          n_min=0.0,
          n_max=7.0,
          x_min=1e-08,
          x_max=11.3,
          p_min=0.0001,
          p_max=5.0,
          solver=2,
          m_grid=array([7.87854850e-05, 2.04238123e-02, 4.08367022e-02, 6.13180882e-02,
                        8.18686128e-02, 1.02488929e-01, 1.23179701e-01, 1.43941602e-01,
                        1.64775317e-01, 1.85681542e-01, 2.06660986e-01, 2.27714368e-01,
                        2.48842419e-01, 2.70045883e-01, 2.91325516e-01, 3.12682087e-01,
                        3.34116378e-01, 3.55629186e-01, 3.77221320e-01, 3.98893603e-01,
                        4.20646875e-01, 4.42481989e-01, 4.64399812e-01, 4.86401230e-01,
                        5.08487143e-01, 5.30658467e-01, 5.52916137e-01, 5.75261103e-01,
                        5.97694334e-01, 6.20216818e-01, 6.42829561e-01, 6.655

In [42]:
models[('DL', '1D', 'sigmoid')].sim.states[...,0].max()

5.2436953

In [43]:
models[('DL', '1D', 'sigmoid')].sim.R 

array(-16.855215, dtype=float32)

In [44]:
models[('DP', '1D')].sim.R 

-19.551780839992176

In [55]:
par = models[('DP', '1D')].par 
sim = models[('DP', '1D')].sim 

beta = par.beta 
beta_t = np.zeros((par.T,sim.N))
for t in range(par.T):
    beta_t[t] = beta**t


print(np.sum(beta_t[...,None]*sim.reward)/sim.N)

-19.551780839992176


In [ ]:
models[('DP', '1D')].vfi.transfer_grid = models[('DP', '1D')].sim.transfer_grid

# a. limits
xlim = [0.1,1000]
ylim = [-100,10]

# b. DP
DP = {
    ('DP',f'1D'):f'DP 1D',
    ('DP',f'2D'):f'DP 2D',
}

# b. DL
specs = {
    ('DL',f'1D','sigmoid'):f'DL 1D',
    ('DL',f'2D','sigmoid'):f'DL 2D',
}

# c. backward
back = []
for D in ('1D','2D'):

    backward = {}

    R = models[('DL',D,'sigmoid')].sim.R
    transfer_beg = compute_transfer(models[('DP',D)].sim.R_transfer,models[('DP', D)].vfi.transfer_grid,R)    
    
    R = models[('DL',D,'backward')].sim.R
    transfer_end = compute_transfer(models[('DP',D)].sim.R_transfer,models[('DP', D)].vfi.transfer_grid,R)    

    backward['transfer_beg'] = transfer_beg
    backward['transfer_end'] = transfer_end

    backward['t_beg'] = models[('DL',D,'sigmoid')].train.K_time
    backward['t_end'] = backward['t_beg'] + models[('DL',D,'backward')].train.K_time

    back.append(backward)

# d. plot
convergence_plot('NonConvexDurables',models,specs,DP=DP,backward=back,do_transfer=True,
                    xlim=xlim,ylim=ylim,legend_ncol=3,
                    folder=folder_save,postfix=f'') 

### 4.1. <a id='toc4_1_'></a>[No backward](#toc0_)

In [ ]:
xlim = [0.1,300]
xlim_low = [0.001,100]
ylim = [-100,10]

for algoname in ['sigmoid']:

    print(algoname)
    

    DP = {
        ('DP',f'1D'):f'DP 1D',
        ('DP',f'2D'):f'DP 2D',
    }

    specs = {
        ('DL',f'1D', f'{algoname}'):f'DL 1D',
        ('DL',f'2D', f'{algoname}'):f'DL 2D',
    }

    convergence_plot('NonConvexDurables',models,specs,DP=DP,do_transfer=True,
                        xlim=xlim,ylim=ylim,
                        folder=folder_save,postfix=f'_{algoname}',DP_name='DP')  


### 4.2. <a id='toc4_2_'></a>[Comparative statics: multiple NNs + FOC](#toc0_)

In [ ]:
xlim = [0.1,300]
xlim_low = [0.001,100]
ylim = [-100,10]
algoname = 'sigmoid'

DP = models[('DP',f'1D')]

specs = {
    ('DL',f'1D', f'{algoname}'):f'DL',
    ('DL',f'1D', f'{algoname}NNs1'):f'DL-NNs1',
    ('DL',f'1D', f'{algoname}FOC'):f'DL-FOC',
}

convergence_plot('NonConvexDurables',models,specs,DP=DP,do_transfer=True,
                 xlim=xlim,ylim=ylim,
                 folder=folder_save,postfix=f'_{algoname}_NNsFOC',DP_name='DP')  


### 4.3. <a id='toc4_3_'></a>[Comparative statics: exploration noise](#toc0_)

In [ ]:
xlim = [0.5,300]
xlim_low = [0.001,100]
ylim = [-10_000,10]
algoname = 'sigmoid'

DP = models[('DP',f'1D')]

specs = {
    ('DL',f'1D', f'{algoname}'):f'DL',
    ('DL',f'1D', f'{algoname}epssigma00expnoise10'):f'DL-no exploration',
    ('DL',f'1D', f'{algoname}epssigma02expnoise11'):f'DL-high exploration',
}

convergence_plot('NonConvexDurables',models,specs,DP=DP,do_transfer=True,
                 xlim=xlim,ylim=ylim,
                 folder=folder_save,postfix=f'_{algoname}_exploration_noise',DP_name='DP')  


## 5. <a id='toc5_'></a>[Euler errors](#toc0_)

In [ ]:
do_display = True
for i,algoname in enumerate(algonames):

    print(algoname)

    fig, axes = plt.subplots(1,2,figsize=(12,4),sharex=True,sharey=True)

    for j,D in enumerate(Ds):
        
        ax = axes[j]
        
        # DP
        DP = models[('DP',f'{D}D')]
        unconstrained = DP.sim.states_pd[:-1,:,0] > 1e-2
        euler_errors = np.abs(DP.sim.euler_error_c[:-1][unconstrained])
        euler_errors_DP = euler_errors[(euler_errors > 0.0)]

        ax.hist(np.log10(euler_errors_DP.flatten()),bins=100,alpha=0.5,
                color='grey',density=True,label=f'DP')
        ax.axvline(x=np.log10(euler_errors_DP.mean()),color='grey',
                   linestyle='--',label=f'DP,mean')

        # DL
        model = models[('DL',f'{D}D',algoname)]
        unconstrained = model.sim.states_pd[:-1,:,0] > 1e-2
        euler_errors = np.abs(model.sim.euler_error_c[:-1][unconstrained])
        euler_errors = euler_errors[euler_errors > 0.0]

        ax.hist(np.log10(euler_errors.flatten()),bins=100,alpha=0.5,
                color=colors[i],density=True,label=f'DL')
        ax.axvline(x=np.log10(euler_errors.mean()),color=colors[i],
                   linestyle='--',label=f'DL,mean')

        ax.set_title(f'{D}D')
        ax.set_xlabel('log10 Euler error')
        ax.set_xlim([-6,0])
        if j == 0:
            ax.set_ylabel('density')
        ax.legend(loc='upper left')

    fig.tight_layout()

    filepath = f"{folder_save}/NonConvexDurablesModel_euler_error_{algoname}.svg"
    fig.savefig(filepath,bbox_inches='tight')

    if do_display:
        plt.show()
    else:
        plt.close(fig)
        display(HTML(f'<a href="{filepath}">{filepath}</a>'))

In [ ]:
D = 1
fig, ax = plt.subplots(1,1,figsize=(6,4),sharex=True,sharey=True)

# DP
DP = models[('DP',f'{D}D')]
unconstrained = DP.sim.states_pd[:-1,:,0] > 1e-2
euler_errors = np.abs(DP.sim.euler_error_c[:-1][unconstrained])
euler_errors_DP = euler_errors[(euler_errors > 0.0)]

ax.hist(np.log10(euler_errors_DP.flatten()),bins=100,alpha=0.5,
        color='grey',density=True,label=f'DP')
ax.axvline(x=np.log10(euler_errors_DP.mean()),color='grey',
            linestyle='--',label=f'DP,mean')

# DL
for i,(algoname,label) in enumerate([('sigmoid','DL'),('sigmoidNNs1','DL-NNs1'),('sigmoidFOC','DL-FOC')]):

    model = models[('DL',f'{D}D',algoname)]
    unconstrained = model.sim.states_pd[:-1,:,0] > 1e-2
    euler_errors = np.abs(model.sim.euler_error_c[:-1][unconstrained])
    euler_errors = euler_errors[euler_errors > 0.0]

    ax.hist(np.log10(euler_errors.flatten()),bins=100,alpha=0.5,
            color=colors[i],density=True,label=label)
    ax.axvline(x=np.log10(euler_errors.mean()),color=colors[i],
                linestyle='--')

ax.set_title(f'{D}D')
ax.set_xlabel('log10 Euler error')
ax.set_xlim([-6,0])
ax.set_ylabel('density')
ax.legend(loc='upper left')

fig.tight_layout()

filepath = f"{folder_save}/NonConvexDurablesModel_euler_error_1D_NNsFOC.svg"
fig.savefig(filepath,bbox_inches='tight')

## 6. <a id='toc6_'></a>[Life-cycle profiles](#toc0_)

### 6.1. <a id='toc6_1_'></a>[1D](#toc0_)

In [ ]:
def plot_with_iqr_lines(ax,arr,label,color,ls='--',marker=None):
    """
    Plot mean and 25th/75th percentile bounds as lines.
    arr: array with shape (T,N),time on axis 0,cross-section on axis 1.
    """

    try:
        arr = arr.detach().cpu().numpy() if torch.is_tensor(arr) else np.asarray(arr)
    except ImportError:
        arr = np.asarray(arr)

    t = np.arange(arr.shape[0])
    mean = np.mean(arr,axis=1)
    p25 = np.percentile(arr,25,axis=1)
    p75 = np.percentile(arr,75,axis=1)

    ax.plot(t,mean,label=label,color=color,ls=ls,marker=marker)
    ax.plot(t,p25,color=color,ls=ls,lw=1,marker=marker,alpha=0.8)
    ax.plot(t,p75,color=color,ls=ls,lw=1,marker=marker,alpha=0.8)


for algoname in algonames:

    print(algoname)

    fig = plt.figure(figsize=(12,12))

    # c
    ax = fig.add_subplot(3,2,1)
    plot_with_iqr_lines(ax,models[('DL','1D',algoname)].sim.c, label='DL',color=colors[0],ls='--',marker='o')
    plot_with_iqr_lines(ax,models[('DP','1D')].sim.c,label='DP',color=colors[1],ls='--')
    ax.set_title('$c$'); ax.set_xlabel('period, $t$'); ax.legend()

    # m
    ax = fig.add_subplot(3,2,2)
    plot_with_iqr_lines(ax,models[('DL','1D',algoname)].sim.states[...,0],label='DL',color=colors[0],ls='--',marker='o')
    plot_with_iqr_lines(ax,models[('DP','1D')].sim.states[...,0],label='DP',color=colors[1],ls='--')
    ax.set_title('$m$'); ax.set_xlabel('period, $t$');

    # d1 
    ax = fig.add_subplot(3,2,3)
    plot_with_iqr_lines(ax,models[('DL','1D',algoname)].sim.d1, label='DL: d1',color=colors[0],ls='--',marker='o')
    plot_with_iqr_lines(ax,models[('DP','1D')].sim.d1,label='DP: d1',color=colors[1],ls='--')
    ax.set_title('$d_1$'); ax.set_xlabel('period, $t$');

    # keeper
    ax = fig.add_subplot(3,2,4)
    ax.plot(np.mean((models[('DL','1D',algoname)].sim.DC==0),axis=1),label='DL: keepers',  color=colors[0],ls='--',marker='o')
    ax.plot(np.mean((models[('DP','1D')].sim.DC==0),axis=1),label='DL: keepers',  color=colors[1],ls='--')
    ax.set_title('share of keeping'); ax.set_xlabel('period, $t$');

    # adj-1
    ax = fig.add_subplot(3,2,5)
    ax.plot(np.mean((models[('DL','1D',algoname)].sim.DC==1),axis=1),label='DL: adjust d1',color=colors[0],ls='--',marker='o')
    ax.plot(np.mean((models[('DP','1D')].sim.DC==1),axis=1),label='DL: adjust d1',color=colors[1],ls='--')
    ax.set_title('share of adjusting'); ax.set_xlabel('period, $t$');

    fig.tight_layout()
    fig.savefig(f'{folder_save}/NonConvexDurablesModel_lcps_{algoname}_1D.svg')

    plt.show()

### 6.2. <a id='toc6_2_'></a>[2D](#toc0_)

In [ ]:
for algoname in algonames:

    print(algoname)

    fig = plt.figure(figsize=(12,12))

    # c
    ax = fig.add_subplot(4,2,1)
    plot_with_iqr_lines(ax, models[('DL','2D',algoname)].sim.c,  label='DL', color=colors[0], ls='--', marker='o')
    plot_with_iqr_lines(ax, models[('DP','2D')].sim.c,label='DP', color=colors[1], ls='--')
    ax.set_title('$c$'); ax.set_xlabel('period, $t$'); ax.legend()

    # m
    ax = fig.add_subplot(4,2,2)
    plot_with_iqr_lines(ax, models[('DL','2D',algoname)].sim.states[...,0], label='DL', color=colors[0], ls='--', marker='o')
    plot_with_iqr_lines(ax, models[('DP','2D')].sim.states[...,0],label='DP', color=colors[1], ls='--')
    ax.set_title('$m$'); ax.set_xlabel('period, $t$'); 

    # d1 
    ax = fig.add_subplot(4,2,3)
    plot_with_iqr_lines(ax, models[('DL','2D',algoname)].sim.d1,  label='DL: d1', color=colors[0], ls='--', marker='o')
    plot_with_iqr_lines(ax, models[('DP','2D')].sim.d1,label='DP: d1', color=colors[1], ls='--')
    ax.set_title('$d_1$'); ax.set_xlabel('period, $t$'); 

    # d2
    ax = fig.add_subplot(4,2,4)
    plot_with_iqr_lines(ax, models[('DL','2D',algoname)].sim.d2,  label='DL: d2', color=colors[0], ls='--', marker='o')
    plot_with_iqr_lines(ax, models[('DP','2D')].sim.d2,label='DP: d2', color=colors[1], ls='--')
    ax.set_title('$d_2$'); ax.set_xlabel('period, $t$'); 

    # keeper
    ax = fig.add_subplot(4,2,5)
    ax.plot(np.mean((models[('DL','2D',algoname)].sim.DC==0),axis=1),label='DL: keepers',   color=colors[0], ls='--', marker='o')
    ax.plot(np.mean((models[('DP','2D')].sim.DC==0),axis=1),label='DL: keepers',   color=colors[1], ls='--')
    ax.set_title('share of keeping'); ax.set_xlabel('period, $t$'); 

    # both
    ax = fig.add_subplot(4,2,6)
    ax.plot(np.mean((models[('DL','2D',algoname)].sim.DC==3),axis=1),label='DL: adjust both', color=colors[0], ls='--', marker='o')
    ax.plot(np.mean((models[('DP','2D')].sim.DC==3),axis=1),label='DL: adjust both', color=colors[1], ls='--')
    ax.set_title('share of adjusting - both'); ax.set_xlabel('period, $t$'); 

    # adj-1
    ax = fig.add_subplot(4,2,7)
    ax.plot(np.mean((models[('DL','2D',algoname)].sim.DC==1),axis=1),label='DL: adjust d1', color=colors[0], ls='--', marker='o')
    ax.plot(np.mean((models[('DP','2D')].sim.DC==1),axis=1),label='DL: adjust d1', color=colors[1], ls='--')
    ax.set_title('share of adjusting - $d_1$'); ax.set_xlabel('period, $t$'); 

    # adj-1
    ax = fig.add_subplot(4,2,8)
    ax.plot(np.mean((models[('DL','2D',algoname)].sim.DC==2),axis=1),label='DL: adjust d1', color=colors[0], ls='--', marker='o')
    ax.plot(np.mean((models[('DP','2D')].sim.DC==2),axis=1),label='DL: adjust d1', color=colors[1], ls='--')
    ax.set_title('share of adjusting - $d_2$'); ax.set_xlabel('period, $t$'); 

    fig.tight_layout()
    fig.savefig(f'{folder_save}/NonConvexDurablesModel_lcps_{algoname}_2D.svg')

    plt.show()


### 6.3. <a id='toc6_3_'></a>[Table: Moments](#toc0_)

In [ ]:
# ---------------------------------------------------------------------
# Moments (row-wise,NaN-robust)
# ---------------------------------------------------------------------

def _moments_nanrobust(A,fallback=True):
    '''
    Row-wise mean,median,variance,skewness,kurtosis (non-excess; normal => 3).
    Handles NaNs. If fallback=True,when std==0 or n<3: skew=0,kurt=3.
    '''
    T,N = A.shape

    mu   = np.full(T,np.nan)
    med  = np.full(T,np.nan)
    var_ = np.full(T,np.nan)
    skew = np.full(T,np.nan)
    kurt = np.full(T,np.nan)

    for t in range(T):

        x = A[t]
        m = np.isfinite(x)
        n = m.sum()

        if n == 0:

            if fallback:
                skew[t] = 0.0
                kurt[t] = 3.0
            continue

        xm = x[m]
        mu[t]  = np.mean(xm)
        med[t] = np.median(xm)
        var_[t] = np.var(xm,ddof=0)

        s = np.sqrt(var_[t])
        if s > 0 and n >= 3:
            z = (xm - mu[t]) / s
            skew[t] = np.mean(z**3)
            kurt[t] = np.mean(z**4)  # non-excess kurtosis
        elif fallback:
            skew[t] = 0.0
            kurt[t] = 3.0

    return mu,med,var_,skew,kurt

# ---------------------------------------------------------------------
# Extract simulation variables
# ---------------------------------------------------------------------

def _extract_vars(sim):
    out = {}
    out['m'] = sim.states[:,:,0]
    out['p'] = sim.states[:,:,1]
    for name in ('c','d1','d2'):
        if hasattr(sim,name):
            out[name] = getattr(sim,name)
    return out

# ---------------------------------------------------------------------
# Main table builder
# ---------------------------------------------------------------------

def get_moments_table(models,D,algo,ts=(0,4,9,14,19)):

    if D == '1D':
        vars_list = ['$m$','$c$','$d_1$']
    else:
        vars_list = ['$m$','$c$','$d_1$','$d_2$']

    methods = ['DP','DL']
    moment_names = ['Mean','Median','Variance','Skewness','Kurtosis']

    # rows: (variable,moment)
    rows = pd.MultiIndex.from_product(
        [vars_list,moment_names],
        names=['var','moment']
    )

    # columns: (t,method)
    cols = pd.MultiIndex.from_product(
        [ts,methods],
        names=['t','Model']
    )

    table = pd.DataFrame(index=rows,columns=cols,dtype=float)

    for method in methods:
        key = (method,D) if method == 'DP' else (method,D,algo)
        sim_vars = _extract_vars(models[key].sim)

        for vname in vars_list:
            mu,med,var_,sk,ku = _moments_nanrobust(sim_vars[vname.replace('$','').replace('_','')])

            moments = {
                'Mean': mu,
                'Median': med,
                'Variance': var_,
                'Skewness': sk,
                'Kurtosis': ku,
            }

            for t in ts:
                for mname,arr in moments.items():
                    table.loc[(vname,mname),(t,method)] = arr[t]

    return table


In [ ]:
for D in [f'{D}D' for D in Ds]:
    
    print(D)

    df = get_moments_table(models,D,'sigmoid',ts=(0,4,9,14,19)).round(2)
    display(df)
    
    df.columns.names = ['t = ','Model']

    latex = df.to_latex(
        escape=False,
        na_rep='',
        index=True,
        bold_rows=False,
        column_format='ll' + 'r' * len(df.columns),
        multicolumn=True,
        multicolumn_format='c',
        longtable=False,
        float_format='%.2f',
        header=True,
    )

    latex = (
        latex.replace('\\toprule','\\toprule\n')
            .replace('\\midrule','\\midrule\n')
            .replace('\\bottomrule','\\bottomrule\n')
    )

    latex = latex.replace(
        '\\begin{tabular}',
        '\\centering\n'
        '\\footnotesize\n'
        '\\setlength\\tabcolsep{4pt}\n'
        '\\renewcommand\\arraystretch{1.15}\n'
        '\\begin{tabular}'
    )

    # add spacing after each Kurtosis row (from moments table only)
    latex = re.sub(
        r'^(.*Kurtosis\s*&.*?\\\\)\s*$',
        r'\1\n\\addlinespace[4pt]',
        latex,
        flags=re.M
    )

    filepath = folder_save + f'/NonConvexDurablesModel_moments_{D}_sigmoid.tex'
    with open(filepath,'w',encoding='utf-8') as f:
        f.write(latex)


### 6.4. <a id='toc6_4_'></a>[Tables: Correlations](#toc0_)

In [ ]:
def keep_upper_triangle(table,prefix='Corr',var_order=('m','c','d1','d2','p')):
    '''
    Keep only Corr(x,y) with x < y according to var_order.
    '''

    idx = table.index.to_series()

    pat = rf'^{prefix}\(([^,]+),\s*([^)]+)\)$'
    is_pair = idx.str.match(pat)
    pairs = idx.str.extract(pat)

    order = {v: i for i,v in enumerate(var_order)}
    lx = pairs[0].map(order)
    ly = pairs[1].map(order)

    keep = is_pair & (lx < ly)

    return table.loc[~is_pair | keep].copy()

def get_correlations_table(models,D,algo,ts=(0,4,9,14,19)):

    if D == '1D':
        vars_list = ['$m$','$c$','$d_1$','$p$']
    else:
        vars_list = ['$m$','$c$','$d_1$','$d_2$','$p$']

    methods = ['DP','DL']

    cols = pd.MultiIndex.from_product(
        [ts,methods],
        names=['t','Model']
    )

    table = pd.DataFrame(index=[],columns=cols,dtype=float)

    for method in methods:
        key = (method,D) if method == 'DP' else (method,D,algo)
        sim_vars = _extract_vars(models[key].sim)

        for v1 in vars_list:
            for v2 in vars_list:
                if v1 == v2: continue

                row = f'Corr({v1},{v2})'
                for t in ts:
                    x = sim_vars[v1.replace('$','').replace('_','')][t]
                    y = sim_vars[v2.replace('$','').replace('_','')][t]
                    table.loc[row,(t,method)] = np.corrcoef(x,y)[0,1]

    return keep_upper_triangle(
        table,
        prefix='Corr',
        var_order=('$m$','$c$','$d_1$','$d_2$','$p$')
    )

In [ ]:
for D in [f'{D}D' for D in Ds]:

    print(D)

    df = get_correlations_table(models,D,'sigmoid',ts=(0,4,9,14,19)).round(2)
    display(df)
    
    df.columns.names = ['t = ','Model']

    latex = df.to_latex(
        escape=False,
        na_rep='',
        index=True,
        bold_rows=False,
        column_format='l' + 'r' * len(df.columns),
        multicolumn=True,
        multicolumn_format='c',
        longtable=False,
        float_format='%.2f',
        header=True,
    )

    latex = (
        latex.replace('\\toprule','\\toprule\n')
            .replace('\\midrule','\\midrule\n')
            .replace('\\bottomrule','\\bottomrule\n')
    )

    latex = latex.replace(
        '\\begin{tabular}',
        '\\centering\n'
        '\\footnotesize\n'
        '\\setlength\\tabcolsep{4pt}\n'
        '\\renewcommand\\arraystretch{1.15}\n'
        '\\begin{tabular}'
    )

    filepath = folder_save + f'/NonConvexDurablesModel_correlations_{D}_sigmoid.tex'
    with open(filepath,'w',encoding='utf-8') as f:
        f.write(latex)

## 7. <a id='toc7_'></a>[Cross-section](#toc0_)

In [ ]:
def get_model(models,algoname,D,postfix=None):

    if len(algoname.split('_')) == 2:
        key = (algoname.split('_')[0],f'{D}D',algoname.split('_')[1])
    else:
        key = (algoname,f'{D}D')
    if postfix is not None:
            key = (algoname.split('_')[0],f'{D}D',postfix)
                     
    if key in models:
        return models[key]
    else:
        return None


### 7.1. <a id='toc7_1_'></a>[Scatter](#toc0_)

In [ ]:
def cross_section_scatter(varname, varindex,D,models,N=10_000,ts=None,do_display=True,postfix=None):
    
    ts = [5,10,15] if ts is None else ts

    fig = plt.figure(figsize=(18,6))        
    for i_t,t in enumerate(ts):

        ax = fig.add_subplot(1,len(ts),1+i_t)

        m_base = get_model(models,'DP',D).sim.states[t,:N,varindex]
        m_algo = get_model(models,'DL',D, postfix=postfix).sim.states[t,:N,varindex]
        x_max = max(np.max(m_base),np.max(m_algo)) * 1.05

        # 45 degree
        ax.plot([0,x_max],[0,x_max],color='black',ls='-',lw=3,alpha=0.5,label='45 degree line')
        ax.set_title(f't={t}')

        # poly fit
        a,b,c,d = np.polyfit(m_base,m_algo,3)
        x_fit = np.linspace(0,x_max,1000)
        y_fit = a*x_fit**3 + b*x_fit**2 + c*x_fit + d
        ax.plot(x_fit,y_fit,color=colors[0],ls='-',lw=2,label='best cubic fit')

        # scatter
        ax.scatter(m_base,m_algo,color='black',s=5,label='households',rasterized=True)
        
        # box            
        m_corr = np.corrcoef(m_base,m_algo)[0,1]
        m_abs_diff_rel = np.mean(np.abs((m_algo-m_base)/m_base))
        box_text = ''
        box_text += f'corr.: {m_corr:.3f}\n'
        box_text += f'mean abs. rel. diff: {m_abs_diff_rel:.3f}'
        ax.text(0.95*x_max,0.10*x_max,box_text,fontsize=12,bbox=dict(facecolor='white',alpha=0.75),ha='right')

        # details
        ax.set_xlim([0,x_max])
        ax.set_xlabel(f'DP')
        ax.set_ylim([0,x_max])
        ax.set_ylabel(f'DL')
        ax.legend(loc='upper left')

    fig.tight_layout()
    postfix = f'_{postfix}' if postfix is not None else ''
    filepath = f'{folder_save}/NonConvexDurablesModel_cross_section_scatter_{varname}_{D}D{postfix}.svg'
    fig.savefig(filepath)

    plt.show()

In [ ]:
for algoname in algonames:

    varnames = ['m','n1','n2']
    varindexes = [0,2,3]

    for D in Ds:
        for varname,varindex in zip(varnames,varindexes):

            if D == 1 and varname == 'n2': continue
            print(f'{algoname}, {D}D, {varname}')

            cross_section_scatter(varname,varindex,D,models,do_display=True,postfix=algoname)   
            plt.show()


### 7.2. <a id='toc7_2_'></a>[Histogram](#toc0_)

In [ ]:
def cross_section_histogram(varname,varindex,D,models,N=10_000,ts=None,bins=50,do_display=True,postfix=None):
    """
    Plot histograms of distances to the 45-degree line for each t in ts.
    """
    ts = [5,10,15] if ts is None else ts

    fig,axes = plt.subplots(1,len(ts),figsize=(18,4),sharey=True)
    if len(ts) == 1: axes = [axes]

    for ax,t in zip(axes,ts):

        m_base = get_model(models,'DP',D).sim.states[t,:N,varindex]
        m_algo = get_model(models,'DL',D,postfix=postfix).sim.states[t,:N,varindex]

        distances = (m_algo-m_base)/m_base*100
        distances = np.clip(distances,-5,5)

        ax.hist(distances,bins=bins,alpha=0.7,edgecolor='k',density=True)
        ax.set_xlabel('percent')
        ax.set_title(f't = {t}')

    fig.tight_layout()
    postfix = f'_{postfix}' if postfix is not None else ''
    fig.savefig(f'{folder_save}/NonConvexDurablesModel_cross_section_histogram_{varname}_{D}D{postfix}.svg')

    if do_display: plt.show()

In [ ]:
for algoname in algonames:

    varnames = ['m','n1','n2']
    varindexes = [0,2,3]

    for D in Ds:
        for varname,varindex in zip(varnames,varindexes):

            if (varname == 'n2') and (D == 1): continue

            print(f'{algoname}: {D}D, {varname}')
            cross_section_histogram(varname,varindex,D,models,do_display=True,bins=100,postfix=algoname)   

#### 7.2.1. <a id='toc7_2_1_'></a>[Discrete choices](#toc0_)

#### 7.2.2. <a id='toc7_2_2_'></a>[1D](#toc0_)

In [ ]:
# a. choice matrix DP vs DL (1D, sigmoid)
choice_matrix = np.zeros((2,2))
for i in range(2):
    for j in range(2):
        choice_matrix[i,j] = np.sum(
            (models[('DP','1D')].sim.DC == i) &
            (models[('DL','1D','sigmoid')].sim.DC == j)
        ) / (models[('DL','1D','sigmoid')].sim.N*models[('DL','1D','sigmoid')].par.T)

# b. dataframe with multi-index
row_index = pd.MultiIndex.from_product(
    [['DP'],['Keep','Adjust']],
    names=[None,None]  # no axis names on rows
)

col_index = pd.MultiIndex.from_product(
    [['DL'],['Keep','Adjust']],
    names=[None,'Decision']  # only one "Decision" on the column side
)

choice_df = pd.DataFrame(choice_matrix,index=row_index,columns=col_index)

choice_df = 100 * choice_df  # convert to percent

# c. display and save with styling
styled = (
    choice_df.style
    .set_table_styles(
        [
            # bold ALL row index labels (DP,Keep,Adjust)
            {'selector': 'th.row_heading','props': 'font-weight: bold;'},

            # center top-level column header ("DL") and make it bold
            {'selector': 'th.col_heading.level0','props': 'text-align: center; font-weight: bold;'},

            # center second-level column headers ("Keep","Adjust")
            {'selector': 'th.col_heading.level1','props': 'text-align: center;'},
        ]
    ).format("{:.2f}")
)

display(styled)

latex = styled.to_latex(
    hrules=True,
    caption="Choice Matrix: DP vs DL (in %)",
    label="tab:choice_matrix_dp_dl",
)

#### 7.2.3. <a id='toc7_2_3_'></a>[2D](#toc0_)

In [ ]:
# a. choice matrix
choice_matrix = np.zeros((4,4))
for i in range(4):
    for j in range(4):
        choice_matrix[i,j] = np.sum((models[('DP','2D')].sim.DC == i) & (models[('DL','2D','sigmoid')].sim.DC == j)) / (models[('DL','2D','sigmoid')].sim.N * models[('DL','2D','sigmoid')].par.T)

# permutation: to get order Keep,Adj dur 1,Adj dur 2,Adj both
perm = [0,2,1,3]

# b. dataframe with multi-index
choice_matrix_swapped = choice_matrix[np.ix_(perm,perm)]
labels_2d = [
    "Keep",
    "Adjust durable 1",
    "Adjust durable 2",
    "Adjust both durables",
]

labels_2d = [
    "Keep",
    "Adjust durable 1",
    "Adjust durable 2",
    "Adjust both durables",
]

row_index = pd.MultiIndex.from_product(
    [['DP'],labels_2d],
    names=[None,None]
)

col_index = pd.MultiIndex.from_product(
    [['DL'],labels_2d],
    names=[None,'Decision']
)

choice_df_2d = pd.DataFrame(choice_matrix_swapped,index=row_index,columns=col_index)

# c. display and save with styling
choice_df_2d = choice_df_2d * 100

styled_2d = (
    choice_df_2d.style
    .set_table_styles(
        [
            {'selector': 'th.row_heading','props': 'font-weight: bold;'},
            {'selector': 'th.col_heading.level0','props': 'text-align: center; font-weight: bold;'},
            {'selector': 'th.col_heading.level1','props': 'text-align: center;'},
        ]
    )
    .format("{:.2f}")
)

display(styled_2d)

# to save as LaTeX
latex_2d = styled_2d.to_latex(
    hrules=True,
    caption="Choice Matrix for 2D Model: DP vs DL (in %)",
    label="tab:choice_matrix_dp_dl_2d",
)

#### 7.2.4. <a id='toc7_2_4_'></a>[Agreement figure](#toc0_)

In [ ]:
for algoname in algonames:

    print(algoname)

    fig,ax = plt.subplots(nrows=1,ncols=2,figsize=(12,4))

    # count agreements across time
    for i_d,D in enumerate(['1D','2D']):
        agreements_list = []
        for t in range(models[('DP',D)].par.T):
            agreements_t = np.sum(
                models[('DP',D)].sim.DC[t] ==
                models[('DL',D,algoname)].sim.DC[t]
            ) / models[('DL',D,algoname)].sim.N
            agreements_list.append(agreements_t)

        ax[i_d].plot(agreements_list)
        ax[i_d].set_title(f'{D}')
        ax[i_d].set_xlabel('time period t')
        ax[i_d].set_ylabel('share of agreements')

    fig.tight_layout()
    fig.savefig(f'{folder_save}/NonConvexDurablesModel_choice_agreements_over_time_{algoname}.svg')

    plt.show()